# Auto Loader Streaming — Netflix Titles

Baseline incremental ingestion of the ~1000 split files into Bronze using Auto
Loader, with `addNewColumns` schema evolution mode enabled from the start — this
gives the schema-evolution experiment in the next notebook a clean baseline to
build on.

## 1. Configuration

In [0]:
%run ./lab3_00_config

In [0]:
pipeline_name = "netflix_titles_stream"

source_path = f"{storage_root}/ingestion/netflix_stream/"
checkpoint_path = f"{storage_root}/checkpoints/{pipeline_name}/"
schema_location = f"{storage_root}/schema_location/{pipeline_name}/"
bronze_table = f"{catalog}.{bronze_schema}.{pipeline_name}"

print("Source path:", source_path)
print("Checkpoint path:", checkpoint_path)
print("Schema location:", schema_location)
print("Target table:", bronze_table)

## 2. Read with Auto Loader and add metadata columns

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

df_bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

df_bronze_stream = (
    df_bronze_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)

## 3. Write the stream to Bronze

In [0]:
query = (
    df_bronze_stream.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

## 4. Verify row count and schema

In [0]:
row_count = spark.table(bronze_table).count()
print("Row count:", row_count)

spark.table(bronze_table).printSchema()

## 5. Confirm no rescued data on the clean baseline

In [0]:
spark.table(bronze_table).select("_rescued_data").filter("_rescued_data IS NOT NULL").show(5, truncate=False)
